In [1]:
import json
import subprocess
import ollama
import os

In [2]:
model = 'qwen3:1.7b'

SYSTEM_PROMPT = """You are an agent run in the user's terminal. You can list files, read files, write files and run shell commands.
Use your tools to complete assigned tasks, then summarise what you did. The working directory is the folder you launched from."""

messages = []

In [3]:
def read_file(path):
    with open(path, 'r',encoding = "utf-8") as file:
        return file.read()

def list_files(path="."):
    files = []
    for file in os.scandir(path):
        files.append(file.name + ("/" if file.is_dir() else ""))
    return "\n".join(sorted(files) or ("Directory empty"))

def write_file(path,content):
    with open(path,'w',encoding="utf-8") as file:
        file.write(content)
    return f"saved {path} ({len(content)} characters)"

def run_command(command):
    answer = input(f"Run '{command}'? [Y/N]")
    if answer.strip().lower() != 'y':
        return "Command declined"
    result = subprocess.run(
        command,shell=True,capture_output=True,text=True,timeout=120
    )
    output=(result.stdout + result.stderr).strip()
    return output or f"(no output, exit code {result.retruncode})"


TOOLS = {
    "list_files":list_files,
    "read_file":read_file,
    "write_file":write_file,
    "run_command":run_command,
}

TOOL_SCHEMAS = [
    {
        "type":"function",
        "function": {
            "name":"read_file",
            "description":"Read a text file and return contents",
            "parameters":{
                "type":"object",
                "properties":{
                    "path":{"type":"string","description":"path of the file to read"}
                },
                "required":["path"],
            },
        },
    },
        {
        "type":"function",
        "function": {
            "name":"list_files",
            "description":"List files in a directory. Folders end with /.",
            "parameters":{
                "type":"object",
                "properties":{
                    "path":{"type":"string","description":"path of the directory to list"}
                },
                "required":["path"],
            },
        },
    },
        {
        "type":"function",
        "function": {
            "name":"write_file",
            "description":"Create or overwrite a file with given content",
            "parameters":{
                "type":"object",
                "properties":{
                    "path":{"type":"string","description":"path of the file to read"},
                    "content":{"type":"string","description":"content to be written to the file"}
                },
                "required":["path","content"],
            },
        },
    },
        {
        "type":"function",
        "function": {
            "name":"run_command",
            "description":"Run a shell command and report the output. Check for user permission first",
            "parameters":{
                "type":"object",
                "properties":{
                    "command":{"type":"string","description":"Command to run"}
                },
                "required":["command"],
            },
        },
    },
]

def run_tool(tool_call):
    name = tool_call.function.name
    args = tool_call.function.arguments
    print(f"tool:{name}({args})")
    try:
        output = json.dumps(str(TOOLS[name](**args)))
        return output
    except Exception as e:
        return f"Error:{e}"

In [4]:
print(str(TOOLS['list_files']('.')))

.ipynb_checkpoints/
Agent.ipynb
Automate_newsfeed.json
Chinook.db
Joke_Generator.json
MCP_Integration.json
SQLQuery.ipynb
Untitled.ipynb
agent_tools.ipynb
fox.txt
readme.md


In [5]:
def run_agent(messages):
    while True:
        response: ChatResponse = ollama.chat(
            model=model,
            messages=messages,
            tools=TOOL_SCHEMAS,
        )
        messages.append(response.message)
        reply = response.message.content
        if response.message.tool_calls:
            for tc in response.message.tool_calls:
                result = run_tool(tc)
                messages.append({'role': 'tool', 'tool_name': tc.function.name, 'content': str(result)})
        
        else:
            return reply
    

In [6]:
def main():
    messages = [{'role':'system','content':SYSTEM_PROMPT}]
    print("Agent ready. Type 'q' to quit")
    while True:
        user = input("\nYou:")
        if user.strip().lower() in ('q','exit'):
            break
        messages.append({'role':'user','content':user})
        respond = run_agent(messages)
        print(f"\nAgent: {respond}")

if __name__ == "__main__":
    main()

Agent ready. Type 'q' to quit



You: My name is Jacob. How are you today?



Agent: Hello, Jacob! How are you today? I'm doing great! Let me know if there's anything I can assist you with. 😊



You: How many files are in this directory?


tool:list_files({'path': '/home/jacob/Documents'})
tool:run_command({'command': 'ls -R'})


...


You: Write the command to a txt file. Use my name as the title of the file


tool:write_file({'path': '/home/jacob/commands.txt', 'content': 'Jacob\nls -R'})
tool:list_files({'path': '/home/jacob'})
tool:write_file({'content': 'Jacob\nls -R', 'path': '/home/jacob/commands.txt'})

Agent: The file `/home/jacob/commands.txt` cannot be created due to permission issues or path conflicts. However, the command `ls -R` to list all files is already available in the current directory. Let me know if you'd like to execute this command! 📁



You: The issue is a path conflict. Use the '.' directory. The file should be {my_name}.txt, in this directory


tool:write_file({'content': 'Jacob\nls -R', 'path': './Jacob.txt'})

Agent: The file `Jacob.txt` has been successfully saved in the current directory. It contains the following content:
```
Jacob
ls -R
```

Let me know if you'd like to view the file or execute the command! 📁



You: q
